# Assignment 4: End-to-End — Raw File to Model-Ready Data

**Dataset**: Hotel Booking Demand Dataset (Antonio, de Almeida, and Nunes, 2019 / TidyTuesday)

**Source URL**: `https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv`

> **Goal**: Transform a raw, messy dataset with 119,390 rows and 32 columns containing genuine missing values, duplicates, impossible values, inconsistent categories, and data leakage into a clean, leakage-free, model-ready feature matrix without training any model.

## 1. Data Dictionary

| Column Name | Meaning | Data Type | Unit | Expected Range |
| :--- | :--- | :--- | :--- | :--- |
| `hotel` | Type of hotel (Resort Hotel or City Hotel) | Categorical (String) | N/A | {'Resort Hotel', 'City Hotel'} |
| `is_canceled` | Target: whether booking was canceled (1) or not (0) | Binary Integer | Flag | {0, 1} |
| `lead_time` | Number of days between booking entry and arrival | Integer | Days | [0, 737] |
| `arrival_date_year` | Year of arrival date | Integer | Year | [2015, 2017] |
| `arrival_date_month` | Month of arrival date | Categorical (String) | Month name | 12 calendar months |
| `arrival_date_week_number` | Week number of arrival date | Integer | Week | [1, 53] |
| `arrival_date_day_of_month`| Day of the month of arrival date | Integer | Day | [1, 31] |
| `stays_in_weekend_nights` | Number of weekend nights (Sat/Sun) booked | Integer | Nights | [0, 19] |
| `stays_in_week_nights` | Number of weekday nights (Mon–Fri) booked | Integer | Nights | [0, 50] |
| `adults` | Number of adults in the booking | Integer | Count | [1, 26] (0 = impossible) |
| `children` | Number of children in the booking | Float / Integer | Count | [0, 10] |
| `babies` | Number of babies in the booking | Integer | Count | [0, 10] |
| `meal` | Type of meal booked (BB, HB, FB, SC, Undefined) | Categorical (String) | Package code | {'BB', 'HB', 'FB', 'SC', 'Undefined'} |
| `country` | Country of customer origin (ISO 3166-1 alpha-3) | Categorical (String) | ISO code | 178 countries + Unknown |
| `market_segment` | Market segment designation (TA, Direct, etc.) | Categorical (String) | Channel code | {'Direct', 'Corporate', 'Online TA', ...} |
| `distribution_channel` | Booking distribution channel | Categorical (String) | Channel code | {'Direct', 'Corporate', 'TA/TO', ...} |
| `is_repeated_guest` | Whether booking name is from repeated guest | Binary Integer | Flag | {0, 1} |
| `previous_cancellations` | Prior bookings canceled by customer | Integer | Count | [0, 26] |
| `previous_bookings_not_canceled` | Prior bookings not canceled by customer | Integer | Count | [0, 72] |
| `reserved_room_type` | Code of room type reserved | Categorical (String) | Room code | {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'L', 'P'} |
| `assigned_room_type` | Code of room type actually assigned | Categorical (String) | Room code | {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'P'} |
| `booking_changes` | Number of changes made to booking prior to check-in | Integer | Count | [0, 21] |
| `deposit_type` | Deposit type made to guarantee booking | Categorical (String) | Category | {'No Deposit', 'Non Refund', 'Refundable'} |
| `agent` | ID of travel agency that made booking | Numeric / Categorical | Agency ID | [1, 535] (or NULL) |
| `company` | ID of company/entity that made/paid booking | Numeric / Categorical | Company ID | [6, 543] (or NULL) |
| `days_in_waiting_list` | Days the booking was in waiting list | Integer | Days | [0, 391] |
| `customer_type` | Booking customer category | Categorical (String) | Type | {'Transient', 'Contract', 'Transient-Party', 'Group'} |
| `adr` | Average Daily Rate (total cost / total nights) | Float | Currency (€) | [0, 500] (negative = impossible) |
| `required_car_parking_spaces` | Number of car parking spaces required | Integer | Count | [0, 8] |
| `total_of_special_requests` | Number of special requests made (high floor, etc.) | Integer | Count | [0, 5] |
| `reservation_status` | Status: Canceled, Check-Out, No-Show (LEAKAGE) | Categorical (String) | Outcome | {'Canceled', 'Check-Out', 'No-Show'} |
| `reservation_status_date` | Date of last status update (LEAKAGE) | Date (YYYY-MM-DD) | Date | Date string |

## 2. Raw Quality Report
Inspection of the raw dataset reveals six categories of data quality issues that must be resolved prior to pipeline modeling:
1. **Missing values**: `company` (112,593 missing / 94.3%), `agent` (16,340 missing / 13.7%), `country` (488 missing / 0.4%), `children` (4 missing).
2. **Duplicate records**: 31,994 identical duplicate rows exist in the raw file.
3. **Impossible guest count**: 180 rows have `adults == 0`, `children == 0`, and `babies == 0` (zero total guests).
4. **Impossible stay duration**: 715 rows have `stays_in_weekend_nights == 0` and `stays_in_week_nights == 0` (zero nights stay).
5. **Impossible pricing & extreme outliers**: 1 row with negative ADR (`-6.38 €`), and 1 extreme outlier with `adr == 5400.00 €`.
6. **Inconsistent category labels & target leakage**: `meal` has 1,169 instances of `'Undefined'` which is synonymous with `'SC'` (Self Catering); `reservation_status` and `reservation_status_date` directly leak the target variable `is_canceled`.

In [1]:
# Raw Data Quality Profiling
import pandas as pd
import numpy as np

raw_df = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv')

print('=== Raw Data Quality Report ===')
print(f'Total Rows: {len(raw_df):,}')
print(f'Total Columns: {raw_df.shape[1]}')
print(f'Exact Duplicate Rows: {raw_df.duplicated().sum():,}')

# Missing values
missing_series = raw_df.isna().sum()
print('\nMissing values per column:')
print(missing_series[missing_series > 0])

# Impossible / anomalous values
zero_guests = ((raw_df['adults'] + raw_df['children'].fillna(0) + raw_df['babies']) == 0).sum()
zero_stays = ((raw_df['stays_in_weekend_nights'] + raw_df['stays_in_week_nights']) == 0).sum()
negative_adr = (raw_df['adr'] < 0).sum()
extreme_adr = (raw_df['adr'] > 5000).sum()
undefined_meals = (raw_df['meal'] == 'Undefined').sum()

print(f'\nZero-guest bookings: {zero_guests:,}')
print(f'Zero-night stays: {zero_stays:,}')
print(f'Negative ADR values: {negative_adr:,}')
print(f'Extreme ADR outliers (>5000): {extreme_adr:,}')
print(f"Inconsistent 'Undefined' meals: {undefined_meals:,}")

=== Raw Data Quality Report ===
Total Rows: 119,390
Total Columns: 32
Exact Duplicate Rows: 31,994

Missing values per column:
children         4
country        488
agent        16340
company     112593
dtype: int64

Zero-guest bookings: 180
Zero-night stays: 715
Negative ADR values: 1
Extreme ADR outliers (>5000): 1
Inconsistent 'Undefined' meals: 1,169


## 3. Feature Engineering Design

We engineer 5 domain-specific features from the existing raw columns, each justified by operational hotel cancellation dynamics:

1. `total_stay_nights`: Sum of `stays_in_weekend_nights` and `stays_in_week_nights`.
   * *Justification*: Captures the overall duration of the stay as a single continuous metric, as longer trips require higher advance planning and carry greater financial exposure.
2. `total_guests`: Sum of `adults`, `children`, and `babies`.
   * *Justification*: Party size fundamentally affects booking commitment and logistics, with families exhibiting distinct cancellation patterns compared to solo travelers.
3. `room_type_changed`: Binary indicator flag `(reserved_room_type != assigned_room_type).astype(int)`.
   * *Justification*: Operational room discrepancies reflect overbooking or involuntary room downgrades/upgrades, which strongly trigger booking cancellations or customer dissatisfaction.
4. `net_cancellation_history`: `previous_cancellations - previous_bookings_not_canceled`.
   * *Justification*: Measures net historical churn risk by penalizing past cancellations while rewarding verified completed past visits.
5. `has_agent`: Binary indicator flag `(agent > 0).astype(int)`.
   * *Justification*: Distinguishes agency-brokered bookings from direct consumer bookings, capturing differences in commercial cancellation and deposit policies.

In [2]:
# 4. Function 1: load_data()
def load_data():
    """
    Loads the raw hotel bookings dataset from URL or local storage.
    """
    import os
    import pandas as pd

    url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
    if os.path.exists("hotels.csv"):
        return pd.read_csv("hotels.csv")
    return pd.read_csv(url)

In [3]:
# 4. Function 2: clean(df)
def clean(df):
    """
    Cleans the raw DataFrame: drops duplicates, handles missing values,
    removes impossible records, fixes inconsistent categories, and drops data leaks.
    """
    import pandas as pd
    import numpy as np

    cleaned = df.copy()

    # 1. Drop exact duplicate rows
    cleaned = cleaned.drop_duplicates()

    # 2. Drop high-missing (>90%) column and target leakage columns
    leakage_and_sparse_cols = ['company', 'reservation_status', 'reservation_status_date']
    cleaned = cleaned.drop(columns=[c for c in leakage_and_sparse_cols if c in cleaned.columns])

    # 3. Filter impossible / invalid values
    guest_count = cleaned['adults'] + cleaned['children'].fillna(0) + cleaned['babies']
    cleaned = cleaned[guest_count > 0]

    stay_nights = cleaned['stays_in_weekend_nights'] + cleaned['stays_in_week_nights']
    cleaned = cleaned[stay_nights > 0]

    cleaned = cleaned[(cleaned['adr'] >= 0) & (cleaned['adr'] < 5000)]

    # 4. Standardize inconsistent category labels
    cleaned['meal'] = cleaned['meal'].replace({'Undefined': 'SC'})
    cleaned['market_segment'] = cleaned['market_segment'].replace({'Undefined': 'Other'})
    cleaned['distribution_channel'] = cleaned['distribution_channel'].replace({'Undefined': 'Other'})

    # 5. Handle missing values
    cleaned['children'] = cleaned['children'].fillna(0)
    cleaned['country'] = cleaned['country'].fillna('Unknown')
    cleaned['agent'] = pd.to_numeric(cleaned['agent'], errors='coerce').fillna(0).astype(int)

    # Pool high-cardinality country into top 15 + Other
    top_countries = cleaned['country'].value_counts().nlargest(15).index
    cleaned['country'] = cleaned['country'].apply(lambda x: x if x in top_countries else 'Other')

    return cleaned.reset_index(drop=True)

In [4]:
# 4. Function 3: add_features(df)
def add_features(df):
    """
    Engineers domain-relevant features from existing columns.
    """
    df_feat = df.copy()

    # Feature 1: total_stay_nights
    df_feat['total_stay_nights'] = df_feat['stays_in_weekend_nights'] + df_feat['stays_in_week_nights']

    # Feature 2: total_guests
    df_feat['total_guests'] = df_feat['adults'] + df_feat['children'] + df_feat['babies']

    # Feature 3: room_type_changed
    df_feat['room_type_changed'] = (df_feat['reserved_room_type'] != df_feat['assigned_room_type']).astype(int)

    # Feature 4: net_cancellation_history
    df_feat['net_cancellation_history'] = df_feat['previous_cancellations'] - df_feat['previous_bookings_not_canceled']

    # Feature 5: has_agent
    df_feat['has_agent'] = (df_feat['agent'] > 0).astype(int)

    return df_feat

In [5]:
# 4. Function 4: build_pipeline(numeric_cols, categorical_cols)
def build_pipeline(numeric_cols, categorical_cols):
    """
    Constructs a scikit-learn ColumnTransformer Pipeline for imputing, encoding, and scaling.
    """
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import RobustScaler, OneHotEncoder

    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('num', num_pipeline, numeric_cols),
        ('cat', cat_pipeline, categorical_cols)
    ])

    pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
    return pipeline

In [6]:
# 5. Run Full Path from Raw File to X_train, X_test, y_train, y_test
from sklearn.model_selection import train_test_split

# 1. Load
raw_df = load_data()

# 2. Clean
cleaned_df = clean(raw_df)

# 3. Add Features
featured_df = add_features(cleaned_df)

# Separate Target & Features
target_col = 'is_canceled'
X = featured_df.drop(columns=[target_col])
y = featured_df[target_col]

# Remove unneeded high-granularity date columns to avoid temporal memorization
drop_cols = ['arrival_date_year', 'arrival_date_day_of_month']
X = X.drop(columns=[c for c in drop_cols if c in X.columns])

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()

# Stratified 80/20 train/test split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Build and fit pipeline strictly on train set
pipeline = build_pipeline(numeric_cols, categorical_cols)
X_train = pipeline.fit_transform(X_train_raw)
X_test = pipeline.transform(X_test_raw)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape : {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape : {y_test.shape}')

X_train shape: (69309, 85)
X_test shape : (17328, 85)
y_train shape: (69309,)
y_test shape : (17328,)


In [7]:
# 6. Output Cleanliness Assertion Tests
import numpy as np

# Assert no NaN remains in either train or test
assert not np.isnan(X_train).any(), 'Assertion Failed: NaN values detected in X_train!'
assert not np.isnan(X_test).any(), 'Assertion Failed: NaN values detected in X_test!'

# Assert train and test column counts match
assert X_train.shape[1] == X_test.shape[1], 'Assertion Failed: Column count mismatch between train and test!'

# Assert no column is constant
variances = np.var(X_train, axis=0)
assert np.all(variances > 0), 'Assertion Failed: Constant column with zero variance detected!'

print('All 3 assertion tests passed successfully!')
print('✓ No NaN remains in either matrix.')
print(f'✓ Column counts match exactly: {X_train.shape[1]} columns.')
print('✓ No column is constant (all variances > 0).')

All 3 assertion tests passed successfully!
✓ No NaN remains in either matrix.
✓ Column counts match exactly: 85 columns.
✓ No column is constant (all variances > 0).


In [8]:
# 7. Before and After Quality Comparison Table
raw_missing = int(raw_df.isna().sum().sum() + (raw_df == 'NULL').sum().sum())
final_missing = int(np.isnan(X_train).sum() + np.isnan(X_test).sum())

before_after_df = pd.DataFrame({
    'Stage': ['Raw Stage', 'Final Preprocessed Matrix'],
    'Row Count': [len(raw_df), len(X_train) + len(X_test)],
    'Column Count': [raw_df.shape[1], X_train.shape[1]],
    'Missing Value Count': [raw_missing, final_missing],
    'Duplicate Count': [int(raw_df.duplicated().sum()), 0]
})

print(before_after_df.to_string(index=False))

                    Stage  Row Count  Column Count  Missing Value Count  Duplicate Count
                Raw Stage     119390            32               129425            31994
Final Preprocessed Matrix      86637            85                    0                0


In [9]:
# 8. Save Cleaned Data and Fitted Pipeline
import joblib

# Save cleaned DataFrame as CSV
cleaned_df.to_csv('cleaned_hotel_bookings.csv', index=False)
print('Saved cleaned_hotel_bookings.csv successfully.')

# Save fitted pipeline
joblib.dump(pipeline, 'pipeline.joblib')
print('Saved pipeline.joblib successfully.')

Saved cleaned_hotel_bookings.csv successfully.
Saved pipeline.joblib successfully.


## 9. Cleaning & Preprocessing Decisions

- `dropped exact duplicates`: 31,994 identical duplicate rows removed to prevent data leakage and artificially inflated model accuracy.
- `dropped company`: 94.3% missing (112,593 missing / NULL values out of 119,390 rows) making imputation statistically meaningless.
- `dropped reservation_status and reservation_status_date`: 100% target leakage, as check-out and cancellation states are recorded strictly post-event.
- `removed zero-guest bookings`: 180 rows where adults + children + babies == 0, representing impossible reservations.
- `removed zero-night stays`: 715 rows where stays_in_weekend_nights + stays_in_week_nights == 0, representing ghost or canceled-on-entry records.
- `filtered negative ADR`: 1 row with adr = -6.38 €, representing an impossible accounting reversal error.
- `filtered extreme ADR outlier`: 1 row with adr = 5400.00 €, representing a severe data entry error.
- `consolidated meal 'Undefined' into 'SC'`: 1,169 rows merged because both represent Self-Catering without catering packages.
- `standardized 'Undefined' in market_segment and distribution_channel`: 7 rows reassigned to 'Other' to avoid single-instance dummy column creation.
- `imputed missing children with 0`: 4 missing values filled using the median/mode of zero.
- `imputed missing country with 'Unknown'`: 488 missing values categorized to avoid losing valuable booking rows.
- `collapsed country cardinality to top 15 + 'Other'`: compressed 178 countries into 16 categories, preventing high-dimensional one-hot sparsity.
- `applied RobustScaler on numeric features`: chosen because heavy right-skewed variables (e.g. `lead_time`, `previous_cancellations`) resist standard z-score normalization without outlier distortion.

## 10. Persistent Dataset Flaws That Could Not Be Fixed

Despite rigorous cleaning and leakage removal, three fundamental data flaws remain unfixable without additional upstream data collection:
1. **Lack of Unique Customer Identifiers**: Because the dataset contains no hashed customer IDs, we cannot track customer lifetimes, multi-hotel loyalty, or individual booking frequency over time; customer history is only captured via crude aggregate counters (`is_repeated_guest`, `previous_cancellations`).
2. **Missing Macroeconomic and Competitor Pricing Context**: The dataset covers a specific Portuguese economic window (2015–2017) without external flight prices, currency exchange rates, consumer confidence indices, or regional competitor rates, making it difficult for models to distinguish seasonal macro-demand shifts from individual hotel cancellation drivers.
3. **Heavy Anonymization of Commercial Intermediaries**: The `agent` and `company` fields are completely anonymized numeric integers with high cardinality and high missingness, obscuring critical B2B terms, deposit agreements, and negotiated corporate cancellation flexibilities that strongly govern whether a booking is canceled in practice.